# Week 9 — PPO · Workbook

Eight graded exercises. Reference impls live in `project.solutions` and `project.train`.


**Teaching companion:** [Worked examples, derivations, analogies, and self-checks](teaching_guide.md). Read the relevant example before attempting its exercise.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, torch, matplotlib.pyplot as plt
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
from project import solutions as sol
from project.train import train


## Exercise 1 (★) — Implement and probe PPO-clip

Verify that `ppo_clip_loss(logp, logp_old, adv)` equals `-ratio * adv` when
ratio is in $[1-\varepsilon,1+\varepsilon]$. Check both advantage signs: saturation occurs only above the upper threshold for positive advantage and below the lower threshold for negative advantage.

> **Hint**: try `logp = log(1.5)`, `logp_old = 0`, $\varepsilon = 0.2$.


In [ ]:
logp = torch.log(torch.tensor([1.5]))
logp_old = torch.zeros(1)
for adv_val in [+1.0, -1.0]:
    adv = torch.tensor([adv_val])
    L = sol.ppo_clip_loss(logp, logp_old, adv, clip_eps=0.2).item()
    print(f'adv={adv_val:+}  L_CLIP={L:+.3f}')
# Expected: adv=+1 -> ratio capped at 1.2, L = -1.2;  adv=-1 -> uncapped at 1.5, L = +1.5
assert abs(sol.ppo_clip_loss(logp, logp_old, torch.tensor([1.0]), 0.2).item() + 1.2) < 1e-5


## Exercise 2 (★) — Where PPO-clip is non-differentiable

The clip function is non-differentiable at $\rho \in \{1-\varepsilon, 1+\varepsilon\}$.
Why does this not matter for SGD? (Answer in a markdown cell.)


In [ ]:
# It is a measure-zero set; auto-grad picks one of the subgradients (Pytorch picks 1).
# Empirically, gradients pass through `clamp` cleanly outside [1-eps, 1+eps] as 0,
# and as ratio inside. The non-smoothness is just kink, not pathology.
print('answered')


## Exercise 3 (★★) — Value clipping

Implement value clipping and verify it is the same as MSE when `v == v_old`,
and never smaller than the corresponding half-MSE. It is strictly larger only when the clipped prediction has a larger squared error; moving outside the window is not sufficient.


In [ ]:
v = torch.tensor([1.0, 2.0, 5.0])
v_old = torch.tensor([1.0, 2.0, 3.0])
ret = torch.tensor([1.5, 2.5, 4.0])
L_clipped = sol.ppo_value_clip_loss(v, v_old, ret, 0.2).item()
L_mse = 0.5 * ((v - ret) ** 2).mean().item()
print(f'L_clip = {L_clipped:.3f},  L_mse = {L_mse:.3f}')
assert L_clipped >= L_mse


## Exercise 4 (★★) — Train PPO on BipedalWalker

Use a short budget. The learning curve should trend upward; we check that the
final 20-episode window is at least 20 above the initial 20-episode window.


In [ ]:
out = train(seed=0, n_updates=30, num_envs=4, rollout_steps=128,
            n_epochs=4, minibatch_size=64)
r = np.array(out.episode_returns) if out.episode_returns else np.zeros(1)
print('n_episodes:', len(r))
if len(r) >= 40:
    initial = float(r[:20].mean()); final = float(r[-20:].mean())
    print('initial 20-mean =', round(initial,1), 'final 20-mean =', round(final,1))
fig, ax = plt.subplots(figsize=(5.5, 3.0))
ax.plot(r, color=RL_PALETTE['teal']); ax.set_xlabel('episode'); ax.set_ylabel('return')
fig.tight_layout(); plt.show()


## Exercise 5 (★★) — Ablating implementation details

Train two short PPO runs, with and without `use_advantage_norm`, and compare.


In [ ]:
from project.train import train
out_full = train(seed=0, n_updates=15, num_envs=2, rollout_steps=64,
                 n_epochs=2, minibatch_size=32)
out_noad  = train(seed=0, n_updates=15, num_envs=2, rollout_steps=64,
                  n_epochs=2, minibatch_size=32, use_advantage_norm=False)
r_full = float(np.mean(out_full.episode_returns[-10:])) if out_full.episode_returns else 0.0
r_noad = float(np.mean(out_noad.episode_returns[-10:])) if out_noad.episode_returns else 0.0
print('full last-10:', round(r_full,1), '  no adv-norm last-10:', round(r_noad,1))


## Exercise 6 (★★) — Track the approximate KL

After each update, the KL between $\pi_\theta$ and $\pi_{\theta_\text{old}}$ is recorded.
Plot it. The target was 0.02; with KL early-stopping enabled, the curve should stay roughly bounded.


In [ ]:
out = train(seed=0, n_updates=20, num_envs=2, rollout_steps=64,
            n_epochs=4, minibatch_size=32)
kls = [m['approx_kl'] for m in out.update_metrics]
fig, ax = plt.subplots(figsize=(5.5, 2.8))
ax.plot(kls, color=RL_PALETTE['orange']); ax.axhline(0.02, color=RL_PALETTE['gray'], ls=':')
ax.set_xlabel('update'); ax.set_ylabel('approx KL'); fig.tight_layout(); plt.show()
print('max KL observed:', round(max(kls), 4))


## Exercise 7 (★★★) — Adaptive KL penalty (PPO-KL variant)

Implement the adaptive-penalty update rule and demonstrate how `beta` evolves.


In [ ]:
beta = 1.0; target_kl = 0.01
trajectory = []
for kl in [0.05, 0.04, 0.001, 0.0005, 0.02, 0.015]:
    beta = sol.adaptive_kl_penalty_update(beta, kl, target_kl)
    trajectory.append((kl, beta))
for kl, b in trajectory:
    print(f'kl={kl:.4f} -> beta={b:.3f}')


## Exercise 8 (★★★, *meaningful objective*) — RLHF-style KL penalty

Add a reference-policy KL term $-\beta\, \KL(\pi_\theta \| \pi_\text{ref})$ to the
reward, with $\pi_\text{ref}$ the policy at iteration $0$. Show that the new policy
stays within a small KL ball of $\pi_\text{ref}$ even when we deliberately corrupt
the reward signal.

Here we *simulate* the test: take a Gaussian policy, add an aligned KL penalty, and
verify that maximising the surrogate keeps the KL bounded.


In [ ]:
# Gaussian over a single action: pi_theta = N(mu, 1), pi_ref = N(0, 1).
mu = torch.tensor(0.0, requires_grad=True)
opt = torch.optim.Adam([mu], lr=0.1)
beta = 0.5
for step in range(50):
    # 'corrupted' reward says always push mu higher (proxy reward)
    proxy_reward = mu  # linear in action mean -- a runaway proxy
    kl_to_ref = 0.5 * mu ** 2  # KL(N(mu,1)||N(0,1))
    loss = -(proxy_reward - beta * kl_to_ref)
    opt.zero_grad(); loss.backward(); opt.step()
print('final mu:', round(mu.item(),3), 'final KL_to_ref:', round(0.5*mu.item()**2,3))
# With beta = 0.5, the optimum sits at mu = 1/beta = 2 (gradient zero); KL = 2.
# Without the penalty, mu would diverge.
assert abs(mu.item()) < 3.0


### Recap

You implemented PPO-clip and value-clip, trained on BipedalWalker, audited the
37 implementation details, and previewed the RLHF/KL-penalty pattern that we'll
fully exploit in week 18.


<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 (★) — Implement and probe PPO-clip</summary>

> **Hint**: try `logp = log(1.5)`, `logp_old = 0`, $\varepsilon = 0.2$.

</details>

<details>
<summary>Exercise 2 (★) — Where PPO-clip is non-differentiable</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 3 (★★) — Value clipping</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 4 (★★) — Train PPO on BipedalWalker</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 5 (★★) — Ablating implementation details</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 6 (★★) — Track the approximate KL</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 7 (★★★) — Adaptive KL penalty (PPO-KL variant)</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 8 (★★★, *meaningful objective*) — RLHF-style KL penalty</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

After solving, change one input or assumption and predict the result before
running it. If a cell calls `project.solutions`, passing it verifies the
reference demonstration; test your own implementation separately.
